# Camada Silver - Brazilian E-Commerce (Olist) + Meteorologia INMET (Enzo)

Limpa as tabelas Bronze e grava as tabelas Silver em Delta, na base `silver` do grupo.

| Tabela Silver | 1 linha = | Usada no modelo em |
|---|---|---|
| `silver.olist_itens` | 1 item de encomenda (`order_id` + `order_item_id`) com compra entre junho e dezembro de 2017 | `fact_itens_olist` |
| `silver.olist_vendedores` | 1 vendedor | `dim_vendedor` |
| `silver.olist_avaliacoes` | 1 avaliação (`review_id`) criada entre junho e dezembro de 2017 | `fact_avaliacoes` (fonte Olist) |
| `silver.inmet_meteorologia_diaria` | 1 estado do Brasil × 1 dia, entre junho e dezembro de 2017 | `fact_meteorologia_diaria` |

Regras do grupo aplicadas:
- **Intervalo comum:** 2017-06-01 a 2017-12-31 (data da compra nos itens, data de criação nas avaliações, data da observação no INMET).
- Encomendas canceladas e indisponíveis (`canceled`, `unavailable`) ficam de fora.
- `dias_entrega`, `dias_atraso` e `is_atrasada` são da **encomenda** e repetem-se em cada item: contar com `COUNT(DISTINCT order_id)`.
- A categoria do produto passa para inglês com a tabela de tradução da Olist (entra depois na `mapa_categorias` do grupo).
- Precipitação diária: **soma** das leituras de cada estação no dia e, depois, média entre as estações do estado.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from delta import *

# Stack comum do grupo: HDFS (hdfs-nn), Hive Metastore e Delta Lake 2.4
warehouse_location = "hdfs://hdfs-nn:9000/warehouse"

builder = SparkSession \
    .builder \
    .appName("Silver - Olist + INMET (Enzo)") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.session.timeZone", "UTC") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [2]:
# Intervalo de tempo comum do grupo (README): só entram registos de 1 de junho a 31 de dezembro de 2017
DATA_INICIO = "2017-06-01"
DATA_FIM = "2017-12-31"

In [3]:
spark.sql("CREATE DATABASE IF NOT EXISTS silver LOCATION 'hdfs://hdfs-nn:9000/warehouse/silver.db/'")

def gravar_silver(df, tabela):
    df.write \
        .mode("overwrite") \
        .format("delta") \
        .option("overwriteSchema", "true") \
        .option("path", f"hdfs://hdfs-nn:9000/warehouse/silver.db/{tabela}/") \
        .saveAsTable(f"silver.{tabela}")
    print(f"silver.{tabela}: {spark.table(f'silver.{tabela}').count()} linhas")

def nulos(df):
    """Número de nulos por coluna."""
    df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show(vertical=True)

def texto(c):
    """Remove espaços nas pontas e passa texto vazio a NULL."""
    t = F.trim(F.col(c))
    return F.when(t == "", None).otherwise(t)

## Encomendas, clientes e produtos

In [4]:
def ts(c):
    return F.to_timestamp(F.trim(F.col(c)), "yyyy-MM-dd HH:mm:ss")

orders = spark.table("bronze.olist_orders").select(
    texto("order_id").alias("order_id"), texto("customer_id").alias("customer_id"),
    F.lower(texto("order_status")).alias("estado_encomenda"),
    ts("order_purchase_timestamp").alias("data_hora_compra"),
    ts("order_delivered_customer_date").alias("data_hora_entrega"),
    ts("order_estimated_delivery_date").alias("data_hora_entrega_prevista"))
print("Encomendas:", orders.count(), "| order_id duplicados:", orders.count() - orders.select("order_id").distinct().count())
orders.groupBy("estado_encomenda").count().orderBy(F.desc("count")).show()

customers = spark.table("bronze.olist_customers").select(
    texto("customer_id").alias("customer_id"),
    texto("customer_zip_code_prefix").alias("cep_cliente"),          # texto: mantém os zeros à esquerda
    F.initcap(texto("customer_city")).alias("cidade_cliente"),
    F.upper(texto("customer_state")).alias("estado_cliente"))

traducao = spark.table("bronze.olist_product_category_name_translation").select(
    texto("product_category_name").alias("categoria_pt"), texto("product_category_name_english").alias("categoria_en"))
products = (spark.table("bronze.olist_products")
    .select(texto("product_id").alias("product_id"), texto("product_category_name").alias("categoria_pt"))
    .join(traducao, "categoria_pt", "left"))
print("Produtos sem categoria:", products.filter(F.col("categoria_pt").isNull()).count(),
      "| categorias sem tradução:", products.filter(F.col("categoria_pt").isNotNull() & F.col("categoria_en").isNull()).select("categoria_pt").distinct().collect())

Encomendas: 99441 | order_id duplicados: 0
+----------------+-----+
|estado_encomenda|count|
+----------------+-----+
|       delivered|96478|
|         shipped| 1107|
|        canceled|  625|
|     unavailable|  609|
|        invoiced|  314|
|      processing|  301|
|         created|    5|
|        approved|    2|
+----------------+-----+

Produtos sem categoria: 610 | categorias sem tradução: [Row(categoria_pt='pc_gamer'), Row(categoria_pt='portateis_cozinha_e_preparadores_de_alimentos')]


In [5]:
encomendas = (orders
    .filter(~F.col("estado_encomenda").isin("canceled", "unavailable"))
    .filter(F.to_date("data_hora_compra").between(DATA_INICIO, DATA_FIM))
    .join(customers, "customer_id", "left")
    .withColumn("data_compra", F.to_date("data_hora_compra"))
    .withColumn("dias_entrega", F.datediff("data_hora_entrega", "data_hora_compra"))
    .withColumn("dias_atraso", F.greatest(F.lit(0), F.datediff("data_hora_entrega", "data_hora_entrega_prevista")))
    .withColumn("is_atrasada", F.when(F.col("data_hora_entrega").isNull(), None)
                                .otherwise(F.col("data_hora_entrega") > F.col("data_hora_entrega_prevista"))))
print("Encomendas no intervalo (sem canceladas/indisponíveis):", encomendas.count())

itens = (spark.table("bronze.olist_order_items")
    .select(texto("order_id").alias("order_id"), F.col("order_item_id").cast("int").alias("order_item_id"),
            texto("product_id").alias("product_id"), texto("seller_id").alias("seller_id"),
            F.col("price").cast("decimal(10,2)").alias("preco"), F.col("freight_value").cast("decimal(10,2)").alias("frete"))
    .join(encomendas, "order_id", "inner")
    .join(products, "product_id", "left")
    .select("order_id", "order_item_id", "data_compra", "seller_id", "product_id", "categoria_pt", "categoria_en",
            "customer_id", "cidade_cliente", "estado_cliente", "estado_encomenda", "preco", "frete",
            "dias_entrega", "dias_atraso", "is_atrasada"))

assert itens.count() == itens.select("order_id", "order_item_id").distinct().count(), "item repetido"
assert itens.filter(F.col("preco") < 0).count() == 0
print("Itens:", itens.count(), "| encomendas:", itens.select("order_id").distinct().count())
nulos(itens)
gravar_silver(itens, "olist_itens")

Encomendas no intervalo (sem canceladas/indisponíveis): 33240
Itens: 37987 | encomendas: 33236
-RECORD 0---------------
 order_id         | 0   
 order_item_id    | 0   
 data_compra      | 0   
 seller_id        | 0   
 product_id       | 0   
 categoria_pt     | 607 
 categoria_en     | 608 
 customer_id      | 0   
 cidade_cliente   | 0   
 estado_cliente   | 0   
 estado_encomenda | 0   
 preco            | 0   
 frete            | 0   
 dias_entrega     | 673 
 dias_atraso      | 0   
 is_atrasada      | 673 

silver.olist_itens: 37987 linhas


In [6]:
vendedores = (spark.table("bronze.olist_sellers")
    .select(texto("seller_id").alias("seller_id"), texto("seller_zip_code_prefix").alias("cep_vendedor"),
            F.initcap(texto("seller_city")).alias("cidade"), F.upper(texto("seller_state")).alias("estado"))
    .dropDuplicates(["seller_id"]))
assert vendedores.count() == spark.table("bronze.olist_sellers").count()
gravar_silver(vendedores, "olist_vendedores")

silver.olist_vendedores: 3095 linhas


## Avaliações

A versão anterior lia o CSV sem `multiLine`: os comentários com mudanças de linha partiam-se em linhas a mais
(101 746 em vez de 99 224). A Bronze já é lida corretamente. Aqui valida-se o domínio da nota (1 a 5) e trata-se
o `review_id` repetido: a mesma avaliação aparece ligada a mais de uma encomenda do mesmo cliente; fica uma linha
por avaliação, ligada à primeira encomenda.

In [7]:
rev = spark.table("bronze.olist_order_reviews").select(
    texto("review_id").alias("review_id"), texto("order_id").alias("order_id"),
    F.col("review_score").cast("int").alias("nota"),
    ts("review_creation_date").alias("data_hora_criacao"), ts("review_answer_timestamp").alias("data_hora_resposta"))

print("Registos:", rev.count(), "| review_id distintos:", rev.select("review_id").distinct().count())
print("Notas fora de 1-5 ou em falta:", rev.filter(~F.col("nota").between(1, 5) | F.col("nota").isNull()).count())
rev.groupBy("nota").count().orderBy("nota").show()

w = Window.partitionBy("review_id").orderBy("order_id")
avaliacoes = (rev
    .filter(F.col("nota").between(1, 5))
    .withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")
    .withColumn("data_criacao", F.to_date("data_hora_criacao"))
    .filter(F.col("data_criacao").between(DATA_INICIO, DATA_FIM))
    .join(orders.select("order_id", "customer_id"), "order_id", "left")
    .join(customers.select("customer_id", "estado_cliente"), "customer_id", "left")
    .withColumn("dias_resposta", F.datediff("data_hora_resposta", "data_hora_criacao"))
    .withColumn("is_negativa", F.col("nota") <= 2)
    .select("review_id", "order_id", "data_criacao", "estado_cliente", "nota", "is_negativa", "dias_resposta"))

assert avaliacoes.count() == avaliacoes.select("review_id").distinct().count()
print("Avaliações no intervalo:", avaliacoes.count())
nulos(avaliacoes)
gravar_silver(avaliacoes, "olist_avaliacoes")

Registos: 99224 | review_id distintos: 98410
Notas fora de 1-5 ou em falta: 0
+----+-----+
|nota|count|
+----+-----+
|   1|11424|
|   2| 3151|
|   3| 8179|
|   4|19142|
|   5|57328|
+----+-----+

Avaliações no intervalo: 32487
-RECORD 0-------------
 review_id      | 0   
 order_id       | 0   
 data_criacao   | 0   
 estado_cliente | 0   
 nota           | 0   
 is_negativa    | 0   
 dias_resposta  | 0   

silver.olist_avaliacoes: 32487 linhas


## Meteorologia (INMET)

In [8]:
obs = spark.table("bronze.inmet_observacoes").select(
    texto("CodigoOMM").alias("estacao"),
    F.upper(texto("Estado")).alias("estado"),
    F.to_date(F.trim(F.col("Data")), "dd/MM/yyyy").alias("data"),
    texto("Hora").alias("hora"),
    F.col("Precipitacao").cast("double").alias("precipitacao"),
    F.col("TempBulboSeco").cast("double").alias("temperatura"),
    F.col("UmidadeRelativa").cast("double").alias("humidade"))
print("Observações:", obs.count(), "| datas inválidas:", obs.filter(F.col("data").isNull()).count())

obs = obs.filter(F.col("data").between(DATA_INICIO, DATA_FIM))
print("Observações no intervalo:", obs.count())

# Leituras repetidas (mesma estação, dia e hora)
antes = obs.count()
obs = obs.dropDuplicates(["estacao", "data", "hora"])
print("Leituras repetidas removidas:", antes - obs.count())

# Valores impossíveis passam a NULL (não se eliminam as outras medidas da mesma leitura)
obs = (obs
    .withColumn("precipitacao", F.when(F.col("precipitacao") < 0, None).otherwise(F.col("precipitacao")))
    .withColumn("temperatura", F.when(~F.col("temperatura").between(-10, 50), None).otherwise(F.col("temperatura")))
    .withColumn("humidade", F.when(~F.col("humidade").between(0, 100), None).otherwise(F.col("humidade"))))

# 1.º por estação e dia: precipitação = soma das leituras; temperatura e humidade = média
estacao_dia = obs.groupBy("estado", "estacao", "data").agg(
    F.sum("precipitacao").alias("precipitacao_dia"),
    F.avg("temperatura").alias("temperatura_dia"),
    F.avg("humidade").alias("humidade_dia"))

# 2.º por estado e dia: média entre estações
meteo = (estacao_dia.groupBy("estado", "data").agg(
        F.round(F.avg("precipitacao_dia"), 2).cast("decimal(8,2)").alias("precipitacao_media_diaria_mm"),
        F.round(F.avg("temperatura_dia"), 2).cast("decimal(6,2)").alias("temperatura_media"),
        F.round(F.avg("humidade_dia"), 2).cast("decimal(6,2)").alias("humidade_media"),
        F.countDistinct("estacao").alias("n_estacoes"))
    .orderBy("estado", "data"))

assert meteo.count() == meteo.select("estado", "data").distinct().count()
print("Estados:", meteo.select("estado").distinct().count(), "| linhas estado x dia:", meteo.count())
nulos(meteo)
gravar_silver(meteo, "inmet_meteorologia_diaria")

Observações: 1845665 | datas inválidas: 0
Observações no intervalo: 126980
Leituras repetidas removidas: 0
Estados: 26 | linhas estado x dia: 5544
-RECORD 0---------------------------
 estado                       | 0   
 data                         | 0   
 precipitacao_media_diaria_mm | 12  
 temperatura_media            | 17  
 humidade_media               | 17  
 n_estacoes                   | 0   

silver.inmet_meteorologia_diaria: 5544 linhas


## Resultado

In [9]:
spark.sql("""
    SELECT date_format(data_compra, 'yyyy-MM') AS mes, COUNT(*) AS itens, COUNT(DISTINCT order_id) AS encomendas,
           ROUND(SUM(preco), 0) AS vendas,
           ROUND(COUNT(DISTINCT CASE WHEN is_atrasada THEN order_id END) / COUNT(DISTINCT order_id) * 100, 1) AS pct_encomendas_atrasadas
    FROM silver.olist_itens GROUP BY 1 ORDER BY 1
""").show()
spark.sql("""
    SELECT nota, COUNT(*) AS avaliacoes FROM silver.olist_avaliacoes GROUP BY nota ORDER BY nota
""").show()
spark.sql("""
    SELECT estado, COUNT(*) AS dias, ROUND(AVG(precipitacao_media_diaria_mm), 2) AS chuva_media_mm, ROUND(AVG(temperatura_media), 1) AS temp_media
    FROM silver.inmet_meteorologia_diaria GROUP BY estado ORDER BY estado
""").show(30)

+-------+-----+----------+-------+------------------------+
|    mes|itens|encomendas| vendas|pct_encomendas_atrasadas|
+-------+-----+----------+-------+------------------------+
|2017-06| 3571|      3205| 429917|                     3.8|
|2017-07| 4493|      3946| 492287|                     3.4|
|2017-08| 4882|      4272| 568246|                     3.3|
|2017-09| 4815|      4227| 621416|                     5.1|
|2017-10| 5300|      4547| 660180|                     5.2|
|2017-11| 8626|      7421|1003862|                    14.1|
|2017-12| 6300|      5618| 742184|                     8.2|
+-------+-----+----------+-------+------------------------+

+----+----------+
|nota|avaliacoes|
+----+----------+
|   1|      3498|
|   2|      1019|
|   3|      2665|
|   4|      6425|
|   5|     18880|
+----+----------+

+------+----+--------------+----------+
|estado|dias|chuva_media_mm|temp_media|
+------+----+--------------+----------+
|    AC| 213|          4.83|      26.4|
|    AL| 214|   

In [10]:
spark.stop()